# Genai Analysis

Research code accompanying [Şeker (2026), Expert Systems](https://doi.org/10.1111/exsy.70300).
See the [repository README](../README.md) for dependencies, dataset preparation and citation.

This is an exploratory notebook: select the intended dataset and experiment cells before
running. Local intermediate datasets and manual notebook state are required. Stored
outputs and execution counts have been removed. Model and API identifiers retain the
historical experiment settings; check availability before calling external services.

Publication edits: notebook outputs/metadata cleared; setup documentation and working directory updated; `tdqm` corrected to `tqdm`; saved dataset paths placed under `data/`. The original Google LLC copyright and Apache-2.0 notice are retained below.


In [ ]:
from pathlib import Path
import os

# Keep relative data/model paths anchored at the repository root.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)


In [ ]:
# @title Copyright 2025 Google LLC. (the "License");
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

The new **[Google Gen AI SDK](https://github.com/googleapis/python-genai)** provides a unified interface to [Gemini 2.0](https://ai.google.dev/gemini-api/docs/models/gemini-v2) through both the [Gemini Developer API](https://ai.google.dev/gemini-api/docs) and the Gemini API

This notebook will walk you through:

* [Installing and setting-up](Get_started.ipynb#scrollTo=Mfk6YY3G5kqp) the Google GenAI SDK
* [Text](Get_started.ipynb#scrollTo=6TYNPrNvQ8ue)
* Counting [tokens](Get_started.ipynb#scrollTo=_9B8pb7tv_Cx)
* Setting system instructions
* Configuring [safety filters](Get_started.ipynb#scrollTo=HTAnYx_bbxPk)
* [Controlling generated output](Get_started.ipynb#scrollTo=nyZMoM6tgnTA)
* Sending [asynchronous](Get_started.ipynb#scrollTo=plCtEIaHuv96) requests
* Generating [text embeddings](Get_started.ipynb#scrollTo=sXNCRn8Wx71d)
* Using [context caching](Get_started.ipynb#scrollTo=oTgeR3_9wN5J)

More details about this new SDK on the [documentation](https://ai.google.dev/gemini-api/docs/sdks).

## Setup

### Install SDK

Install the SDK from [PyPI](https://github.com/googleapis/python-genai).

In [ ]:
%pip install -q tqdm
%pip install -q datasets

In [ ]:
%pip install -U -q 'google-genai'
%pip install python-dotenv

### Setup API key

Copy `.env.example` to `.env` at the repository root and set `GOOGLE_API_KEY`. The next cell reads it with `python-dotenv`.


In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Set the Google API key
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")


### Initialize SDK client

With the new SDK you now only need to initialize a client with you API key (or OAuth if using [Vertex AI](https://cloud.google.com/vertex-ai)). The model is now set in each call.

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GOOGLE_API_KEY)

### Choose a model

* [supported models](https://ai.google.dev/gemini-api/docs/models)

In [ ]:
#MODEL_ID = "gemini-2.0-flash"
MODEL_ID = "gemini-1.5-flash"

# Configure model parameters

* used the [Configure safety filters](https://ai.google.dev/gemini-api/docs/safety-settings) to prevent dangerous results.
* used the [controlled generation](https://ai.google.dev/gemini-api/docs/structured-output?lang=python#generate-json) capability in Gemini API to constraint the model output to a structured format.
* used the `count_tokens` method to calculate the number of input tokens before sending a request to the Gemini API. Not sure if google charging those requests (docs doesnt contain any information about that)
* More details in [doc](https://googleapis.github.io/python-genai/genai.html#genai.types.HarmBlockThreshold)

In [ ]:
safety_settings = [
    types.SafetySetting(
        category="HARM_CATEGORY_DANGEROUS_CONTENT", # "HARM_CATEGORY_HARASSMENT", "HARM_CATEGORY_HATE_SPEECH", "HARM_CATEGORY_SEXUALLY_EXPLICIT","HARM_CATEGORY_CIVIC_INTEGRITY"
        threshold="BLOCK_ONLY_HIGH",
    ),
]

# Yeni Yöntem
* Kullanacağımız bu

In [ ]:
system_instruction = """
You are an advanced language analysis engine designed to process customer product reviews. Follow below work steps on a Review:
Work Steps:

1.⁠ Translation: 
    - Translate the review into English.

2. Sentiment Analysis:
    - Classify the emotional tone of the review as one of the following categories:
        - Joy
        - Dissatisfaction (Mild negativity due to unmet needs, not angry or emotionally broken.)
        - Disappointment (Strong emotional letdown that often involves regret or sadness.)
        - Anger
        - Mixed (Consists of both positive and negative sentiments.)
        - Neutral
    - Give the sentiment score in range of -1 to 1.

3.⁠ ⁠Topic Analysis:
    - Extract a specific topic that clearly defines user's main issue or experience.
    - Limit the topic to no more than 4 words.  
    - If the topic of review cannot be extract, return an empty string: "".

"""

In [ ]:
system_instruction_en = """
You are an advanced language analysis engine designed to process customer product reviews. Follow below work steps on a Review:
Work Steps:

1. Sentiment Analysis:
    - Classify the emotional tone of the review as one of the following categories:
        - Joy
        - Dissatisfaction (Mild negativity due to unmet needs, not angry or emotionally broken.)
        - Disappointment (Strong emotional letdown that often involves regret or sadness.)
        - Anger
        - Mixed (Consists of both positive and negative sentiments.)
        - Neutral
    - Give the sentiment score in range of -1 to 1.

2.⁠ ⁠Topic Analysis:
    - Extract a specific topic that clearly defines user's main issue or experience.
    - Limit the topic to no more than 4 words.  
    - If the topic of review cannot be extract, return an empty string: "".

"""

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from google import genai
from google.genai import types
from pydantic import BaseModel
import json
import time
import pandas as pd
import random
from collections import deque
from tqdm import tqdm

class Data(BaseModel):
    translated_text: str
    emotion: str
    sentiment_score: float
    topic: str

class Data_en(BaseModel):
    emotion: str
    sentiment_score: float
    topic: str

class RateLimiter:
    def __init__(self, max_calls, period = 60):
        self.max_calls = max_calls
        self.period = period
        self.call_times = deque()
        self.lock = threading.Lock()

    def allow(self):
        """
        Checks if a call is allowed under the current rate limit.

        Returns:
            bool: True if the call is allowed, False otherwise.
        """
        with self.lock:
            current_time = time.time()
            # Remove outdated calls
            while self.call_times and current_time - self.call_times[0] > self.period:
                self.call_times.popleft()
            if len(self.call_times) < self.max_calls:
                self.call_times.append(current_time)
                return True
            return False

def translate(text: str, system_instruction, data ) -> dict:
    """Blocking call to GenAI; returns a dict with language, translated_text, used_tokens."""

    prompt = f""" Review: {text}

    """

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(
            safety_settings=safety_settings,
            system_instruction=system_instruction,
            temperature=0.3, 
            top_p=0.75, 
            top_k=10,
            candidate_count=1,
            presence_penalty=1,
            frequency_penalty=1,
            response_mime_type='application/json',
            response_schema=data,
        )
    )
    json_data = json.loads(response.text)
    # Rename keys
    json_data['gemini_input_tokens'] = getattr(response.usage_metadata, 'prompt_token_count', None)
    json_data['gemini_output_tokens'] = getattr(response.usage_metadata, 'candidates_token_count', None)

    if not json_data:
        raise RuntimeError("Empty response from genai")
    return json_data

# Fefaultly it wait 60s with 1s period for the available token in the bucket
def available_token_in_bucket(limiter, duration=60):
    """
    Waits until the RateLimiter allows a new call, or timeout occurs.

    Args:
        limiter (RateLimiter): The rate limiter instance.
        duration (int): Maximum time in seconds to wait for an available token.

    Returns:
        bool: True if a token became available within the duration, False otherwise.
    """
    start = time.time()

    while not limiter.allow():
        if time.time() - start > duration:
            return False
        time.sleep(0.5)
    return True

def call_genai(df: pd.DataFrame) -> list:
    """
    ⚠️  IMPORTANT: The DataFrame['text'] must be cleaned from non-str values before use!
    -------------------------------------------------------------------
    Processes up to 100 concurrent translation requests, ensuring no more than
    2000 API calls are made per 60 seconds.

    Args:
        df (pd.DataFrame): DataFrame with a 'text' column containing input strings.

    Returns:
        list: List of dictionaries with translation results and metadata, one per row.
    """
    period = 70
    limiter = RateLimiter(max_calls =2000, period=period)

    # placeholder for results
    results = [None] * len(df)

    def rate_limited_calls(row, idx) -> dict:
        if not available_token_in_bucket(limiter, period):
              return {
                    'gemini_language': None,
                    'translated_text': None,
                    'gemini_input_tokens': None,
                    'gemini_output_tokens': None,
                    'error': f"The call_genai().rate_limited_calls().available_token_in_bucket() is returned False for row: {idx}"
                }
        # Check if the text is in English
        if row.language == "en":
            # Use the English system instruction
            data = Data_en
            sys_inst = system_instruction_en
        else:
            # Use the translation system instruction
            data = Data
            sys_inst = system_instruction

        return translate(row.text, sys_inst, data)

    # sends pack of 100 requests with avarage pack respsonse time as 3s, but each will block if we're over 2000req/60s (100*20)request/(3*20)s
    with ThreadPoolExecutor(max_workers=100) as executor:
        future_to_idx = {
            executor.submit(rate_limited_calls, row, idx): idx
            for idx, row in enumerate(df.itertuples())
        }
        for future in  tqdm(as_completed(future_to_idx), total=len(future_to_idx), desc="Sentiment analysis in progress"): 
            idx = future_to_idx[future]
            try:
                res = future.result()
            except Exception as e:
                print(f"❗ Error translating row {idx}: {e}")
                res = {
                    'gemini_language': None,
                    'translated_text': None,
                    'gemini_input_tokens': None,
                    'gemini_output_tokens': None,
                    'error': str(e)
                }
            results[idx] = res

    return results

In [ ]:
from datasets import load_from_disk

dataset = load_from_disk("data/MARC_with_languages")
df = dataset.to_pandas()

In [ ]:
from datasets import load_from_disk

dataset = load_from_disk("data/MABSA_with_languages")
df = dataset.to_pandas()

RESET INDEX OF DF !!!!!!!

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
# execution time 121s for 25 rows
translation_result = call_genai(df) # IF SOMETHING GOES WRONG, YOU CAN ACCESS THE RESULTS IN THE GLOBAL results LIST

In [ ]:
translations_df = pd.DataFrame(translation_result) # results list to DataFrame
#translations_df.to_csv("translation_result_marc.csv", index=False)
translations_df.to_csv("translation_result_mabsa.csv", index=False)

In [ ]:
translations_df.info()

In [ ]:
from datasets import Dataset

# Pandas df → Huggingface Dataset'e dönüştür
dataset = Dataset.from_pandas(df)

# Dataset olarak kaydet
#dataset.save_to_disk('data/MARC_translate_sentiment_topic')
dataset.save_to_disk('data/MABSA_translate_sentiment_topic')

In [ ]:
ress = pd.concat([df, translations_df], axis=1)

In [ ]:
ress.info()

In [ ]:
ress.emotion.value_counts()

In [ ]:
ress.stars.value_counts()

In [ ]:
ress.topic.value_counts().iloc[:20]

In [ ]:
ress.gemini_input_tokens.sum()

In [ ]:
ress.gemini_output_tokens.sum()


In [ ]:
((362044641.0 / 1000000)*0.075) + ((87897363.0 / 1000000)*0.3)*45

In [ ]:
((362044641.0 / 1000000)*0.15) + ((87897363.0 / 1000000)*0.6)*45

In [ ]:
((362044641.0 / 1000000)*0.10) + ((87897363.0 / 1000000)*0.4)*45

In [ ]:
for(category,text,score,emotion) in zip(translated_df['product_category'], translated_df['translated_text'], translated_df['sentiment_score'], translated_df['emotion']):
    print(f"Category: {category}\n Text: {text}\nSentiment Score: {score}\nEmotion: {emotion}\n")

In [ ]:
#hatalı satırları tekrar işlemek 

In [ ]:
retry_df = df[df['error'].notna() & (df['error'].astype(str).str.strip().str.lower() != 'none')].copy()


In [ ]:
retry_df = retry_df.reset_index().rename(columns={"index": "original_index"})

In [ ]:
retry_df = df[df['error'].apply(lambda x: x is not None and str(x).strip().lower() != 'none')].copy()
retry_df = retry_df.reset_index().rename(columns={"index": "original_index"})

In [ ]:
results

In [ ]:
results = call_genai(retry_df)

In [ ]:
df_results = pd.DataFrame(results)
df_results['original_index'] = retry_df['original_index'].values

# Sadece güncellenen alanlar
update_cols = ['translated_text', 'emotion', 'sentiment_score', 'topic', 'gemini_input_tokens', 'gemini_output_tokens']

# Ana df'ye güncelleme
for col in update_cols:
    df.loc[df_results['original_index'], col] = df_results[col].values

In [ ]:
print("✅ Hatalı satırların güncellenme süreci tamamlandı.")
print(f"Boş kalan hata sayısı: {(df['error'].notna() & (df['error'] != 'None')).sum()}")


In [ ]:
import numpy as np
df.loc[df_results['original_index'], 'error'] = np.nan

In [ ]:
print("Güncel hata sayısı:", df['error'].notna().sum())  # Artık 0 ya da başka kalanlar olmalı
print("NaN olan satır sayısı:", df['error'].isna().sum())  # 482341’e yakın olmalı


In [ ]:
from datasets import Dataset

# Pandas df → Huggingface Dataset'e dönüştür
dataset = Dataset.from_pandas(df)

# Dataset olarak kaydet
#dataset.save_to_disk('data/MARC_translate_sentiment_topic')
dataset.save_to_disk('data/MABSA_translate_sentiment_topic_all')